# Reaction path and transition state with AFIR + MLFF (MACE)

Discover a reaction path and its transition state locally with ASE. An artificial force (AFIR) pulls two selected atoms together while a MACE foundation model supplies energies and forces, so bonds break and form during the optimization.

Example: the Claisen rearrangement of allyl vinyl ether (C5H8O) to 4-pentenal, a concerted [3,3]-sigmatropic shift in which a C–O bond breaks and a C–C bond forms. The barrier of ~30 kcal/mol is too high to cross by relaxation.

The inputs are the molecule, the atom pair to push together and the force ramp. The product, the barrier and the transition state are results, not inputs: `PRODUCT_NAME` only labels what the search produces and is not checked against it.

<h2 style="color:green">Usage</h2>

1. Set the molecule and the MACE model in cell 1.2 (or use the default values).
1. Run the cells up to 2.1: it lists the atoms with their neighbours and reports the pairs currently selected. Nothing before section 3 uses the model, so this takes seconds.
1. Set the atom pairs in 1.2 from that listing. AFIR pushes exactly these atoms together, so indices left over from another molecule search a different reaction.
1. Click "Run" > "Run All" to run the remaining cells.
1. Wait for the search to finish.
1. Scroll down to view the discovered path, the transition state and the energy diagram.

## Summary

2. Load the molecule from the uploads folder or from PubChem.
3. Relax it with MACE to get the reactant.
4. Ramp an artificial force between the target atoms until the molecule reacts.
5. Remove the bias to recover the physical energy profile. Its maximum is the first estimate of the transition state.
6. Relax the end of the path to get the product.
7. Refine that estimate to a saddle point with the dimer method.
8. Verify the saddle point has one imaginary frequency.
9. Follow that mode in both directions to identify the minima it connects.
10. Plot the energy diagram and view the structures.
11. Save the structures, the energy profile and the plots.

Runs locally through ASE, without platform compute or authentication.

## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
from mat3ra.notebooks_utils.mlff import get_mlff_install_profiles
from mat3ra.notebooks_utils.packages import install_packages

await install_packages(get_mlff_install_profiles("mace"))

from mat3ra.notebooks_utils.pyodide.packages.patches import apply_all_patches

apply_all_patches("mace")

### 1.2. Set parameters

Atom indices refer to the loaded structure and are listed by cell 2.1. AFIR adds a bias term $E_\text{bias} = \alpha \, r_{ij}$ between the atoms of `BOND_FORMING_PAIR`: a constant attractive force $\alpha$. Too small a force stops in a biased minimum, too large a force distorts the geometry, so it is ramped, each stage starting from the structure the previous one converged to.

In [ ]:
# Input structure: a file named after the molecule in FOLDER, otherwise fetched from PubChem by that name
FOLDER = "../../uploads"
MOLECULE_NAME = "allyl vinyl ether"

# Atom indices, as listed by cell 2.1
BOND_FORMING_PAIR = (4, 5)  # the only pair AFIR pulls together; this choice selects the reaction
BOND_BREAKING_PAIR = (0, 1)  # expected to come apart; also directs the saddle search. None if not known
REPORTED_PAIRS = {"carbonyl": (0, 3)}  # further distances to follow, {} for none

# Artificial force between the target atoms, ramped stage by stage until the molecule reacts (eV/Å)
AFIR_FORCE_RAMP = [1.0, 2.0, 3.0, 4.0]

# Name of the expected product, used to label results. None if unknown
PRODUCT_NAME = "4-pentenal"
# Measured activation energy to draw on the diagram (kcal/mol). None to leave it out
EXPERIMENTAL_ACTIVATION_ENERGY = 30.6

# What the MACE model was trained on: "organic" for molecules, "inorganic" for crystals and surfaces
MACE_MODEL_FAMILY = "organic"
MACE_MODEL = "medium"  # choose between "small", "medium" and "large"
MACE_DISPERSION = False  # D3 dispersion correction, not needed for this intramolecular rearrangement
MACE_DEFAULT_DTYPE = "float64"  # float64 is recommended for geometry optimization and vibrations
MACE_DEVICE = "cpu"  # hardware target: "cpu" or "cuda" (GPU)

### 1.3. Convergence thresholds and output paths

These set precision and run time, not which reaction is found. Each search writes to its own folder, named after the molecule and the pair pushed together, so several molecules or several pairs can be explored side by side.

In [ ]:
# Maximum force on any atom at convergence (eV/Å)
RELAXATION_FMAX = 0.03
AFIR_FMAX = 0.05
SADDLE_FMAX = 0.02

AFIR_MAX_STEPS_PER_STAGE = 150
AFIR_MAX_DISPLACEMENT = 0.1  # per-step displacement cap, keeps the biased path smooth (Å)
SADDLE_MAX_STEPS = 200
MODE_FOLLOWING_MAX_STEPS = 400

# Modes below this magnitude are the translations and rotations of a free molecule (cm⁻¹)
IMAGINARY_MODE_THRESHOLD = 50
# Displacement along the imaginary mode used to leave the saddle point (Å)
REACTION_MODE_DISPLACEMENT = 0.3
# Two relaxed structures count as different minima if a tracked distance differs by more than this (Å)
MINIMUM_SEPARATION = 0.5
# Scales covalent radii when deciding which atoms count as bonded, for the listing in 2.1
BOND_TOLERANCE = 1.1

# Distances followed along the path and reported for every structure
TRACKED_PAIRS = {"forming": BOND_FORMING_PAIR}
if BOND_BREAKING_PAIR:
    TRACKED_PAIRS["breaking"] = BOND_BREAKING_PAIR
TRACKED_PAIRS.update(REPORTED_PAIRS)

PRODUCT_LABEL = PRODUCT_NAME or "product"
SEARCH_NAME = f"{MOLECULE_NAME.replace(' ', '_')}_{BOND_FORMING_PAIR[0]}-{BOND_FORMING_PAIR[1]}"
RESULTS_FOLDER = f"results/{SEARCH_NAME}"  # one folder per search
AFIR_TRAJECTORY_PATH = f"{RESULTS_FOLDER}/afir_path.traj"

## 2. Load the molecule
### 2.1. Read from uploads, or fetch the 3D structure from PubChem

PubChem provides an optimized 3D conformer for most small molecules, so any molecule name is a valid starting point.

The connectivity listing below is what the atom indices in 1.2 refer to: hydrogens are summarized as a count, so each heavy atom is identified by its neighbours. In this molecule atoms 4 and 5 are the only carbons with a single carbon neighbour and two hydrogens, i.e. the two terminal CH2 groups, and atom 1 is the CH2 attached to the ether oxygen.

The selected pairs are then resolved against the structure and reported, so indices left over from another molecule show up here rather than after the search.

In [ ]:
import io
import os
from urllib.parse import quote

from ase.io import read, write
from ase.neighborlist import NeighborList, natural_cutoffs

PUBCHEM_STRUCTURE_URL = "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{name}/SDF?record_type=3d"


def fetch_pubchem_structure(name):
    url = PUBCHEM_STRUCTURE_URL.format(name=quote(name))
    try:
        from pyodide.http import open_url

        return open_url(url).read()
    except ImportError:
        from urllib.request import urlopen

        return urlopen(url).read().decode()


molecule_path = os.path.join(FOLDER, MOLECULE_NAME.replace(" ", "_") + ".xyz")

if not os.path.exists(molecule_path):
    write(molecule_path, read(io.StringIO(fetch_pubchem_structure(MOLECULE_NAME)), format="sdf"))
    print(f"Fetched {MOLECULE_NAME} from PubChem, saved to {molecule_path}")

molecule = read(molecule_path)
symbols = molecule.get_chemical_symbols()

neighbor_list = NeighborList(natural_cutoffs(molecule, mult=BOND_TOLERANCE), self_interaction=False, bothways=True)
neighbor_list.update(molecule)
neighbors_of = {index: sorted(int(n) for n in neighbor_list.get_neighbors(index)[0]) for index in range(len(molecule))}
selected_pairs = {pair: label for label, pair in TRACKED_PAIRS.items()}


def describe(index):
    hydrogen_count = sum(1 for neighbor in neighbors_of[index] if symbols[neighbor] == "H")
    return f"{index}:{symbols[index]}" + (f" +{hydrogen_count}H" if hydrogen_count else "")


print(f"{MOLECULE_NAME}: {molecule.get_chemical_formula()}\n")
print(f"{'index':>5}  {'atom':<5}bonded to")
for index, symbol in enumerate(symbols):
    if symbol == "H":
        continue
    heavy_neighbors = ", ".join(f"{n}:{symbols[n]}" for n in neighbors_of[index] if symbols[n] != "H")
    hydrogen_count = sum(1 for n in neighbors_of[index] if symbols[n] == "H")
    bonded = heavy_neighbors + (f" + {hydrogen_count}H" if hydrogen_count else "")
    role = next((f"{label} pair {pair}" for pair, label in selected_pairs.items() if index in pair), "")
    print(f"{index:>5}  {symbol:<5}{bonded:<22}{role}")

out_of_range = {label: pair for label, pair in TRACKED_PAIRS.items() if max(pair) >= len(molecule)}
if out_of_range:
    raise ValueError(
        f"{out_of_range} out of range for {MOLECULE_NAME}, which has {len(molecule)} atoms. "
        "Set the pairs in 1.2 from the listing above."
    )

print()
for label, pair in TRACKED_PAIRS.items():
    connection = f"{describe(pair[0])} — {describe(pair[1])}"
    print(f"{label:<10}{connection:<24}{molecule.get_distance(*pair):>8.2f} Å")

if any(symbols[index] == "H" for index in BOND_FORMING_PAIR):
    print("\n⚠️ The forming pair includes a hydrogen: intended for a hydrogen transfer, otherwise pick two heavy atoms.")
if BOND_FORMING_PAIR[1] in neighbors_of[BOND_FORMING_PAIR[0]]:
    print("\n⚠️ The forming pair is already bonded. AFIR pushes together atoms that are not bonded yet.")

### 2.2. View the molecule

In [ ]:
from mat3ra.made.tools.build_components import MaterialWithBuildMetadata
from mat3ra.made.tools.convert import from_ase
from mat3ra.notebooks_utils.ipython.entity.material.visualize import ViewersEnum, visualize_materials as visualize

VACUUM = 5.0  # padding around the molecule, in Å, so it can be handled as a material


def to_material(atoms, name):
    boxed_atoms = atoms.copy()
    boxed_atoms.center(vacuum=VACUUM)
    material = MaterialWithBuildMetadata.create(from_ase(boxed_atoms))
    material.name = name
    return material


visualize([{"material": to_material(molecule, MOLECULE_NAME), "title": MOLECULE_NAME}], viewer=ViewersEnum.wave)

## 3. Relax the reactant with MACE
### 3.1. Create the ASE calculator

Models are loaded from the ones shipped with the platform (`packages/models`), so nothing is downloaded at run time and local and JupyterLite runs use the same weights.

`MACE_MODEL_FAMILY = "organic"` selects MACE-OFF23, trained on organic molecules, distributed under the Academic Software License, which does not permit commercial use. `"inorganic"` selects MACE-MP-0, trained on Materials Project crystals, MIT licensed.

With `"inorganic"` this reaction closes the C–C bond without breaking the C–O bond and ends at a cyclic structure, and section 7 reports that no transition state was found. Energies of the two families use different references and are not comparable.

In [ ]:
from mat3ra.notebooks_utils.mlff import create_mlff_calculator
from mat3ra.notebooks_utils.pyodide.packages.mace import MODEL_FAMILY_LABELS

MACE_MODEL_LABEL = f"{MODEL_FAMILY_LABELS[MACE_MODEL_FAMILY]} ({MACE_MODEL})"

calculator = create_mlff_calculator(
    "mace",
    {
        "family": MACE_MODEL_FAMILY,
        "model": MACE_MODEL,
        "dispersion": MACE_DISPERSION,
        "default_dtype": MACE_DEFAULT_DTYPE,
        "device": MACE_DEVICE,
    },
)

### 3.2. Relax the reactant

In [ ]:
from ase.optimize import BFGS

# ASE sends optimizer logs to /dev/null when no logfile is given, and Pyodide cannot flush it; a buffer works in both
OPTIMIZER_LOG = io.StringIO()

reactant = molecule.copy()
reactant.calc = calculator

BFGS(reactant, logfile=OPTIMIZER_LOG).run(fmax=RELAXATION_FMAX)
reactant_energy = reactant.get_potential_energy()

print(f"Relaxed {MOLECULE_NAME}: {reactant_energy:.3f} eV")

## 4. Push the reaction with an artificial force

The bias is applied as an ASE `ExternalForce` constraint, which is the AFIR term: a constant force $\alpha$ along the vector between the target atoms, with energy $\alpha \, r_{ij}$. Physical forces come from MACE, so which bond breaks in response is determined by the model.

In [ ]:
from ase.constraints import ExternalForce
from ase.io.trajectory import Trajectory

structure = reactant.copy()
structure.calc = calculator

os.makedirs(RESULTS_FOLDER, exist_ok=True)
trajectory = Trajectory(AFIR_TRAJECTORY_PATH, "w", structure)
trajectory.write()

for artificial_force in AFIR_FORCE_RAMP:
    structure.set_constraint(ExternalForce(*BOND_FORMING_PAIR, -artificial_force))
    BFGS(structure, trajectory=trajectory, maxstep=AFIR_MAX_DISPLACEMENT, logfile=OPTIMIZER_LOG).run(
        fmax=AFIR_FMAX, steps=AFIR_MAX_STEPS_PER_STAGE
    )
    distances = ", ".join(f"{label} = {structure.get_distance(*pair):.2f} Å" for label, pair in TRACKED_PAIRS.items())
    print(f"α = {artificial_force:.1f} eV/Å  →  {distances}")

structure.set_constraint()

## 5. Recover the physical energy landscape
### 5.1. Strip the bias

Each structure along the biased path is re-evaluated without the bias, turning the AFIR trajectory into a physical energy profile. Its maximum is the first estimate of the transition state.

In [ ]:
import numpy as np

EV_TO_KCAL_PER_MOL = 23.060548

images = read(AFIR_TRAJECTORY_PATH, index=":")
unbiased_energies = []
for image in images:
    image.set_constraint()
    image.calc = calculator
    unbiased_energies.append(image.get_potential_energy())

path_energies = (np.array(unbiased_energies) - reactant_energy) * EV_TO_KCAL_PER_MOL
transition_state_guess_index = int(np.argmax(path_energies))

print(f"AFIR path: {len(images)} structures")
print(
    f"Highest point at step {transition_state_guess_index}: "
    f"{path_energies[transition_state_guess_index]:.1f} kcal/mol above the reactant"
)

### 5.2. Plot the discovered path

In [ ]:
from matplotlib import pyplot as plt
from mat3ra.notebooks_utils.plot import display_matplotlib_figure

TRACKED_COLORS = {"forming": "#2e8b57", "breaking": "#c93b3b"}
tracked_distances = {label: [image.get_distance(*pair) for image in images] for label, pair in TRACKED_PAIRS.items()}

path_figure, (energy_axes, distance_axes) = plt.subplots(2, 1, figsize=(8, 7), sharex=True)

energy_axes.plot(path_energies, color="#2b5c8f", linewidth=2)
energy_axes.axvline(transition_state_guess_index, color="#c93b3b", linestyle="--", label="Transition state guess")
energy_axes.set_ylabel("Energy relative to reactant (kcal/mol)")
energy_axes.set_title(f"AFIR path: {MOLECULE_NAME}, {MACE_MODEL_LABEL}")
energy_axes.legend()
energy_axes.grid(True, linestyle=":", alpha=0.6)

for label, distances in tracked_distances.items():
    distance_axes.plot(distances, color=TRACKED_COLORS.get(label), label=f"{label} {TRACKED_PAIRS[label]}")
distance_axes.axvline(transition_state_guess_index, color="#c93b3b", linestyle="--")
distance_axes.set_xlabel("AFIR step")
distance_axes.set_ylabel("Distance (Å)")
distance_axes.legend()
distance_axes.grid(True, linestyle=":", alpha=0.6)

path_figure.tight_layout()
display_matplotlib_figure(path_figure)

## 6. Relax the discovered product

The last structure of the biased path is relaxed without the bias, into the product minimum the search reached.

In [ ]:
product = images[-1].copy()
product.calc = calculator

BFGS(product, logfile=OPTIMIZER_LOG).run(fmax=RELAXATION_FMAX)
reaction_energy = (product.get_potential_energy() - reactant_energy) * EV_TO_KCAL_PER_MOL

print(f"Product energy: {reaction_energy:.1f} kcal/mol relative to the reactant\n")
print(f"{'bond':<20}{'reactant':>12}{'product':>12}")
for label, pair in TRACKED_PAIRS.items():
    print(f"{label + ' ' + str(pair):<20}{reactant.get_distance(*pair):>10.2f} Å{product.get_distance(*pair):>10.2f} Å")

## 7. Refine the transition state

The maximum of the AFIR path lies on a biased trajectory, not on a stationary point of the physical surface: forces there are large and its energy overestimates the barrier. The dimer method follows the lowest-curvature mode uphill and the remaining modes downhill to the nearest first-order saddle point, which defines the activation energy. The search is started along the reaction direction, with the forming pair closing and the breaking pair opening.

In [ ]:
from ase.mep import DimerControl, MinModeAtoms, MinModeTranslate

transition_state = images[transition_state_guess_index].copy()
transition_state.calc = calculator
print(f"Maximum force at the AFIR guess: {np.abs(transition_state.get_forces()).max():.2f} eV/Å")

reaction_direction = np.zeros_like(transition_state.positions)
direction_pairs = [(BOND_FORMING_PAIR, 1.0)] + ([(BOND_BREAKING_PAIR, -1.0)] if BOND_BREAKING_PAIR else [])
for pair, sign in direction_pairs:
    unit_vector = transition_state.positions[pair[1]] - transition_state.positions[pair[0]]
    unit_vector /= np.linalg.norm(unit_vector)
    reaction_direction[pair[0]] += sign * unit_vector
    reaction_direction[pair[1]] -= sign * unit_vector
reaction_direction /= np.linalg.norm(reaction_direction)

dimer_control = DimerControl(
    initial_eigenmode_method="displacement",
    displacement_method="vector",
    logfile=OPTIMIZER_LOG,
    eigenmode_logfile=OPTIMIZER_LOG,
)
dimer = MinModeAtoms(transition_state, dimer_control)
dimer.displace(displacement_vector=0.05 * reaction_direction, mask=[True] * len(transition_state))

MinModeTranslate(dimer, logfile=OPTIMIZER_LOG).run(fmax=SADDLE_FMAX, steps=SADDLE_MAX_STEPS)

transition_state_energy = transition_state.get_potential_energy()
activation_energy = (transition_state_energy - reactant_energy) * EV_TO_KCAL_PER_MOL
saddle_force = float(np.abs(transition_state.get_forces()).max())

print(f"Maximum force at the saddle point: {saddle_force:.3f} eV/Å")
if saddle_force > SADDLE_FMAX:
    print(f"⚠️ Not converged to {SADDLE_FMAX} eV/Å — this structure is not a transition state and the numbers below say nothing about the reaction.")
print(f"Activation energy: {activation_energy:.1f} kcal/mol")
print(", ".join(f"{label} = {transition_state.get_distance(*pair):.2f} Å" for label, pair in TRACKED_PAIRS.items()))

## 8. Verify the transition state

A first-order saddle point has exactly one imaginary frequency, and its mode is the reaction coordinate. A free molecule also has six translational and rotational modes near zero frequency, which appear as small imaginary values; `IMAGINARY_MODE_THRESHOLD` separates them from a reaction mode.

In [ ]:
from ase.vibrations import Vibrations

vibrations = Vibrations(transition_state, name="transition_state_vibrations")
vibrations.run()
vibrations.summary()

frequencies = vibrations.get_frequencies()
imaginary_mode_indices = [
    index
    for index, frequency in enumerate(frequencies)
    if np.iscomplex(frequency) and abs(frequency.imag) > IMAGINARY_MODE_THRESHOLD
]

print(f"\nImaginary modes above {IMAGINARY_MODE_THRESHOLD} cm⁻¹: {len(imaginary_mode_indices)}")
for index in imaginary_mode_indices:
    print(f"  mode {index}: {abs(frequencies[index].imag):.0f}i cm⁻¹")

if not imaginary_mode_indices:
    print("⚠️ No imaginary mode above the threshold — this structure is not a transition state.")

reaction_mode = vibrations.get_mode(imaginary_mode_indices[0] if imaginary_mode_indices else 0)
vibrations.clean()

## 9. Confirm which minima the saddle connects

Displacement along the imaginary mode in both directions, followed by relaxation, identifies the two minima the saddle point joins: one returns to the reactant, the other reaches the product.

In [ ]:
connected_minima = {}

columns = "".join(f"{label:>14}" for label in TRACKED_PAIRS)
print(f"{'direction':<12}{'energy, kcal/mol':>18}{columns}")
for sign, label in ((1.0, "forward"), (-1.0, "reverse")):
    displaced = transition_state.copy()
    displaced.positions += sign * REACTION_MODE_DISPLACEMENT * reaction_mode / np.linalg.norm(reaction_mode)
    displaced.calc = calculator
    BFGS(displaced, logfile=OPTIMIZER_LOG).run(fmax=RELAXATION_FMAX, steps=MODE_FOLLOWING_MAX_STEPS)
    connected_minima[label] = displaced
    energy = (displaced.get_potential_energy() - reactant_energy) * EV_TO_KCAL_PER_MOL
    distances = "".join(f"{displaced.get_distance(*pair):>12.2f} Å" for pair in TRACKED_PAIRS.values())
    print(f"{label:<12}{energy:>18.1f}{distances}")

connects_two_minima = any(
    abs(connected_minima["forward"].get_distance(*pair) - connected_minima["reverse"].get_distance(*pair))
    > MINIMUM_SEPARATION
    for pair in TRACKED_PAIRS.values()
)
print(
    "\n✅ The imaginary mode connects two distinct minima."
    if connects_two_minima
    else "\n⚠️ Both directions relax to the same structure — the saddle does not connect a reactant and a product."
)

## 10. Results
### 10.1. Energy diagram

The experimental activation energy in the gas phase is 30.6 kcal/mol. Foundation models trained on near-equilibrium structures overestimate barriers in the bond-breaking region; reaction energies are reproduced more closely.

In [ ]:
levels = [
    (MOLECULE_NAME, 0.0),
    ("transition state", activation_energy),
    (PRODUCT_LABEL, reaction_energy),
]

diagram_figure, axes = plt.subplots(figsize=(7, 4.5))
axes.plot(range(len(levels)), [energy for _, energy in levels], linestyle="--", color="#999999")
for position, (label, energy) in enumerate(levels):
    axes.hlines(energy, position - 0.25, position + 0.25, color="#2b5c8f", linewidth=4)
    axes.annotate(f"{energy:.1f}", (position, energy), textcoords="offset points", xytext=(0, 10), ha="center")
if EXPERIMENTAL_ACTIVATION_ENERGY:
    axes.axhline(EXPERIMENTAL_ACTIVATION_ENERGY, color="#c93b3b", linestyle=":", label="measured barrier")
axes.set_xticks(range(len(levels)))
axes.set_xticklabels([label for label, _ in levels])
axes.set_ylabel("Energy relative to reactant (kcal/mol)")
axes.set_title(f"{MOLECULE_NAME} → {PRODUCT_LABEL}, {MACE_MODEL_LABEL}")
axes.legend()
axes.grid(True, axis="y", linestyle=":", alpha=0.6)
diagram_figure.tight_layout()
display_matplotlib_figure(diagram_figure)

saddle_note = "" if saddle_force <= SADDLE_FMAX else "  ⚠️ no transition state was found, see 7"
measured = f" (measured: {EXPERIMENTAL_ACTIVATION_ENERGY})" if EXPERIMENTAL_ACTIVATION_ENERGY else ""
print(f"Activation energy: {activation_energy:.1f} kcal/mol{measured}{saddle_note}")
print(f"Reaction energy:   {reaction_energy:.1f} kcal/mol")

### 10.2. View the reactant, transition state and product

In [ ]:
visualize(
    [
        {"material": to_material(reactant, MOLECULE_NAME), "title": MOLECULE_NAME},
        {"material": to_material(transition_state, "Transition state"), "title": "Transition state"},
        {"material": to_material(product, PRODUCT_LABEL), "title": PRODUCT_LABEL},
    ],
    viewer=ViewersEnum.wave,
)

## 11. Save the results
### 11.1. Hand the structures back as materials

Pass the three structures that define the reaction to the environment with `set_materials`.

In [ ]:
from mat3ra.notebooks_utils.material import set_materials

structures = (
    ("reactant", reactant, f"{MOLECULE_NAME}, reactant"),
    ("transition_state", transition_state, f"{MOLECULE_NAME}, transition state"),
    ("product", product, f"{PRODUCT_LABEL} from {MOLECULE_NAME}"),
)

set_materials([to_material(atoms, name) for _, atoms, name in structures], FOLDER)

### 11.2. Write the energy profile and the plots

Write the settings, the resulting energies and the profile behind the plots next to the trajectory already in `RESULTS_FOLDER`.

In [ ]:
import json

results = {
    "reaction": {"reactant": MOLECULE_NAME, "product": PRODUCT_NAME},
    "settings": {
        "calculator": MACE_MODEL_LABEL,
        "tracked_pairs": {label: list(pair) for label, pair in TRACKED_PAIRS.items()},
        "artificial_force_ramp_ev_per_angstrom": AFIR_FORCE_RAMP,
    },
    "activation_energy_kcal_per_mol": round(float(activation_energy), 2),
    "reaction_energy_kcal_per_mol": round(float(reaction_energy), 2),
    "imaginary_frequency_cm": (
        round(float(abs(frequencies[imaginary_mode_indices[0]].imag)), 1) if imaginary_mode_indices else None
    ),
    "transition_state_found": bool(saddle_force <= SADDLE_FMAX and connects_two_minima),
    "distances_angstrom": {
        role: {label: round(float(atoms.get_distance(*pair)), 3) for label, pair in TRACKED_PAIRS.items()}
        for role, atoms, _ in structures
    },
    "afir_path": {
        "transition_state_guess_index": transition_state_guess_index,
        "energy_kcal_per_mol": [round(float(value), 4) for value in path_energies],
        "distance_angstrom": {
            label: [round(float(value), 3) for value in distances] for label, distances in tracked_distances.items()
        },
    },
}

with open(os.path.join(RESULTS_FOLDER, "reaction_path.json"), "w") as file:
    json.dump(results, file, indent=2)

path_figure.savefig(os.path.join(RESULTS_FOLDER, "afir_path.png"), dpi=140)
diagram_figure.savefig(os.path.join(RESULTS_FOLDER, "energy_diagram.png"), dpi=140)

print(f"Saved to {RESULTS_FOLDER}/: " + ", ".join(sorted(os.listdir(RESULTS_FOLDER))))

## References

[1] AFIR method: S. Maeda, K. Morokuma, "Communications: A systematic method for locating transition structures of A+B → X type reactions", J. Chem. Phys. 132, 241102 (2010). https://doi.org/10.1063/1.3457903  
[2] AFIR review: S. Maeda, K. Ohno, K. Morokuma, "Systematic exploration of the mechanism of chemical reactions: the global reaction route mapping (GRRM) strategy", Phys. Chem. Chem. Phys. 15, 3683 (2013). https://doi.org/10.1039/C3CP44063J  
[3] MACE-OFF23 organic foundation models: D. P. Kovács et al., arXiv:2312.15211. https://arxiv.org/abs/2312.15211  
[4] MACE-MP-0 materials foundation model: I. Batatia et al., arXiv:2401.00096. https://arxiv.org/abs/2401.00096  
[5] Dimer method: G. Henkelman, H. Jónsson, "A dimer method for finding saddle points on high dimensional potential surfaces using only first derivatives", J. Chem. Phys. 111, 7010 (1999). https://doi.org/10.1063/1.480097  
[6] Experimental Claisen barrier: F. W. Schuler, G. W. Murphy, "The Kinetics of the Rearrangement of Vinyl Allyl Ether", J. Am. Chem. Soc. 72, 3155 (1950). https://doi.org/10.1021/ja01163a096  
[7] ASE optimizers, constraints and vibrations: https://wiki.fysik.dtu.dk/ase/ase/optimize.html  
[8] PubChem compound "allyl vinyl ether" (CID 221523): https://pubchem.ncbi.nlm.nih.gov/compound/221523  